In [1]:
from huggingface_hub import hf_hub_download
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np
from aeon.classification.interval_based import RSTSF
from sklearn.metrics import accuracy_score
from autotsc.models import StackerV4, LokyStackerV5, LokyStackerV5SoftRF, LokyStackerV6
from autotsc import utils

def load_monash_fold(dataset_name: str, fold: int, test_size: float = 0.2):
    path_X = hf_hub_download(
        repo_id=f"monster-monash/{dataset_name}",
        filename=f"{dataset_name}_X.npy",
        repo_type="dataset",
    )
    path_y = hf_hub_download(
        repo_id=f"monster-monash/{dataset_name}",
        filename=f"{dataset_name}_y.npy",
        repo_type="dataset",
    )
    X = np.load(path_X, mmap_mode="r")
    y = np.load(path_y, mmap_mode="r")

    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=fold)
    train_idx, test_idx = next(sss.split(X, y))

    return X[train_idx], y[train_idx], X[test_idx], y[test_idx]

X_train, y_train, X_test, y_test = load_monash_fold("LakeIce", 7)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)
X_train = X_train.astype(np.float64)
X_test = X_test.astype(np.float64)
print(X_train.dtype)
print(y_train.dtype)

(103424, 1, 161) (103424,) (25856, 1, 161) (25856,)
float64
int64


In [2]:
f = 20
X_train_sub = X_train[::f]
y_train_sub = y_train[::f]
print(X_train_sub.shape, y_train_sub.shape)

(5172, 1, 161) (5172,)


In [3]:
m = LokyStackerV6(random_state=492, n_repetitions=1, n_jobs=16)
m.fit(X_train_sub, y_train_sub)
m.cleanup()

[0.0000s] Starting fitting
[5.6913s] Computed QUANT features in 5.6911s
Starting executor with 16 workers, run_dir=./tscglue/a7d15daff4904c54
[5.6937s] Starting repetition 0
[13.5224s] Computed MultiRocket features in 7.8286s
[21.2340s] Computed Hydra features in 7.7112s
[36.3252s] Computed RDST features in 15.0910s
  Saved quant r0: (5172, 1652) (65.19 MB)
  Saved multirocket r0: (5172, 49728) (1962.23 MB)
  Saved hydra r0: (5172, 5120) (202.03 MB)
  Saved rdst r0: (5172, 30000) (1183.78 MB)
[38.9308s] Feature arrays saved to mmap files
[38.9315s] Starting training with 16 workers for 40 models
[68.1598s] Trained quant-etc in 23.3744s for f-2/r-0 (6.36 MB)
[68.4721s] Trained quant-etc in 23.7408s for f-1/r-0 (6.35 MB)
[68.6030s] Trained quant-etc in 23.9827s for f-0/r-0 (6.32 MB)
[69.6373s] Trained quant-etc in 24.8169s for f-5/r-0 (6.26 MB)
[72.6923s] Trained quant-etc in 27.3261s for f-4/r-0 (6.33 MB)
[72.9094s] Trained quant-etc in 27.5731s for f-3/r-0 (6.38 MB)
[84.6346s] Trained 